# Dataset Prep


## Dependencies


In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install -q torch neuralforecast lightgbm statsforecast datasets
!pip install -q matplotlib seaborn scipy scikit-learn

!pip install -q neuralforecast
!pip install -q git+https://github.com/ydataai/ydata-synthetic.git
!pip install -q tensorflow==2.15.0  # ydata-synthetic backend


!pip install -q torch torchvision
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch: {torch.__version__}")
print(f"GPU    : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

import tensorflow as tf
print(f"TF version : {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print("✅ neuralforecast ready")

print("✅ All packages installed.")

## Configs

In [ ]:
# ============================================================
# CELL 2: Imports & Global Config  (UPDATED)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Experiment config
TEST_MONTHS       = 12     # hold-out horizon
MIN_LENGTH        = 51     # all series are exactly 51 — keeps all
MAX_ZERO_RATIO    = 0.98   # raised from 0.95; max in data is 0.980

# Scarcity regimes — train window = 51 - 12 = 39 timesteps
# Extreme  : first 15 (~38% of 39)
# Moderate : first 20 (~51% of 39)
# Full     : all 39
REGIMES = {
    "extreme": 15,    # was 50 — now ~38% of available train data
    "moderate": 20,   # was 0.25 fraction — now ~51% of train data
    "full": 39,       # was 1.0 fraction — now the exact train length
}

print("✅ Config updated.")
print(f"   Train timesteps available : 51 - {TEST_MONTHS} = {51 - TEST_MONTHS}")
print(f"   Regimes: { {k: v for k, v in REGIMES.items()} }")

## Dataset Loading

In [ ]:
# ============================================================
# CELL 3: Load Car Parts Dataset (Monash Archive via Chronos mirror)
# ============================================================
import datasets as hf_datasets

# Inspect available splits first
ds_info = hf_datasets.load_dataset_builder("autogluon/chronos_datasets", "monash_car_parts")
print("Available splits:", ds_info.info.splits)

# Load whichever split name printed above (likely 'test' or no split needed)
ds = hf_datasets.load_dataset(
    "autogluon/chronos_datasets",
    "monash_car_parts",
)
ds.set_format("numpy")

# Peek at available splits and structure
print("Splits     :", list(ds.keys()))

# Grab whichever split exists
split_name = list(ds.keys())[0]
data = ds[split_name]

print("Num series :", len(data))
print("Features   :", data.features)
print("Sample keys:", list(data[0].keys()))
data[0]

### Dataset parsing

In [ ]:
# ============================================================
# CELL 4 (FIXED): Parse Series into a Usable DataFrame
# ============================================================
records = []
for i, item in enumerate(data):
    values = np.array(item["target"], dtype=np.float32).flatten()
    if len(values) == 0:
        continue
    records.append({
        "series_id"  : i,
        "values"     : values,
        "length"     : len(values),
        "zero_ratio" : float((values == 0).sum()) / len(values),
    })

meta = pd.DataFrame(records)
print(f"Raw series count: {len(meta)}")
print(meta[["length", "zero_ratio"]].describe().round(3))
# ^^^ Check the actual min/median/max length before proceeding

### Debug for Cell 5

In [ ]:
# ============================================================
# CELL 5-DEBUG: Inspect raw records before filtering
# ============================================================
import numpy as np

# Check a few records
for i in range(3):
    row = records[i]
    print(f"\n--- Series {i} ---")
    print(f"  length     : {row['length']}")
    print(f"  zero_ratio : {row['zero_ratio']:.3f}")
    print(f"  nonzero_ct : {(np.array(row['values']) != 0).sum()}")
    print(f"  values type: {type(row['values'])}, dtype: {np.array(row['values']).dtype}")
    print(f"  sample vals: {np.array(row['values'])[:10]}")

# Check distribution of lengths and zero ratios across all series
lengths     = [r['length'] for r in records]
zero_ratios = [r['zero_ratio'] for r in records]
print(f"\nLength    — min:{min(lengths)}  median:{np.median(lengths):.0f}  max:{max(lengths)}")
print(f"ZeroRatio — min:{min(zero_ratios):.3f}  median:{np.median(zero_ratios):.3f}  max:{max(zero_ratios):.3f}")

In [ ]:
# ============================================================
# CELL 5-DIAGNOSE: Why is everything filtered out?
# ============================================================
print("=== Filter breakdown ===")
c1 = (meta["length"] >= MIN_LENGTH).sum()
c2 = (meta["zero_ratio"] <= MAX_ZERO_RATIO).sum()
c3 = (meta["values"].apply(lambda v: (v != 0).sum()) > 1).sum()

print(f"  length >= {MIN_LENGTH}          : {c1} / {len(meta)} pass")
print(f"  zero_ratio <= {MAX_ZERO_RATIO}  : {c2} / {len(meta)} pass")
print(f"  nonzero_count > 1       : {c3} / {len(meta)} pass")

print("\n=== Sample raw values from meta ===")
for i in range(3):
    v = meta.iloc[i]["values"]
    print(f"\n  Series {i}:")
    print(f"    type      : {type(v)}")
    print(f"    dtype     : {getattr(v, 'dtype', 'N/A')}")
    print(f"    length    : {meta.iloc[i]['length']}")
    print(f"    zero_ratio: {meta.iloc[i]['zero_ratio']:.4f}")
    print(f"    first 10  : {v[:10]}")

### Cell 5 continues

In [ ]:
# ============================================================
# CELL 5 (FIXED): Quality Filtering
# ============================================================
meta["nonzero_count"] = meta["values"].apply(lambda v: (v != 0).sum())

mask = (
    (meta["length"] >= MIN_LENGTH) &       # >= 51, keeps all
    (meta["zero_ratio"] <= MAX_ZERO_RATIO) &  # <= 0.98
    (meta["nonzero_count"] > 1)
)

meta_clean = meta[mask].reset_index(drop=True)
print(f"After filtering: {len(meta_clean)} series  "
      f"(removed {len(meta) - len(meta_clean)})")

# Sanity check
print(f"\nZero-ratio breakdown of removed series:")
removed = meta[~mask]
print(f"  Removed due to zero_ratio > 0.98 : {(meta['zero_ratio'] > MAX_ZERO_RATIO).sum()}")
print(f"  Removed due to single spike      : {(meta['nonzero_count'] <= 1).sum()}")

## Visulaising and Exploring Data

In [ ]:
# ============================================================
# CELL 6: Exploratory Data Analysis
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Zero-ratio distribution
axes[0].hist(meta_clean["zero_ratio"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(meta_clean["zero_ratio"].median(), color="red", linestyle="--",
                label=f'Median={meta_clean["zero_ratio"].median():.2f}')
axes[0].set_title("Zero-Inflation Rate")
axes[0].set_xlabel("Fraction of Zeros")
axes[0].legend()

# 2. Series lengths
axes[1].hist(meta_clean["length"], bins=30, color="darkorange", edgecolor="white")
axes[1].set_title("Series Lengths")
axes[1].set_xlabel("Timesteps")

# 3. Spike (non-zero) magnitudes — pooled
all_spikes = np.concatenate([
    v[v > 0] for v in meta_clean["values"]
])
axes[2].hist(np.log1p(all_spikes), bins=50, color="seagreen", edgecolor="white")
axes[2].set_title("log(1+x) Non-Zero Demand")
axes[2].set_xlabel("log(1 + demand)")

plt.suptitle("Car Parts EDA", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nMedian zero-ratio : {meta_clean['zero_ratio'].median():.3f}")
print(f"Median series len : {meta_clean['length'].median():.0f}")
print(f"Spike count total : {len(all_spikes):,}")

## Train/Test Split

In [ ]:
# ============================================================
# CELL 7: Temporal Train/Test Split + Log(1+x) + Z-score
# ============================================================
def make_split(values, test_months=TEST_MONTHS):
    """Returns (train_raw, test_raw) — no transform yet."""
    return values[:-test_months], values[-test_months:]

def preprocess(train_raw, test_raw):
    """log(1+x) → Z-score (fit on train only)."""
    tr = np.log1p(train_raw)
    te = np.log1p(test_raw)
    mu, sigma = tr.mean(), tr.std() + 1e-8
    return (tr - mu) / sigma, (te - mu) / sigma, mu, sigma

splits = []
for _, row in meta_clean.iterrows():
    tr_raw, te_raw = make_split(row["values"])
    tr_sc, te_sc, mu, sigma = preprocess(tr_raw, te_raw)
    splits.append({
        "series_id": row["series_id"],
        "train_raw": tr_raw,
        "test_raw":  te_raw,
        "train_sc":  tr_sc,
        "test_sc":   te_sc,
        "mu": mu, "sigma": sigma,
    })

splits_df = pd.DataFrame(splits)
print(f"✅ Splits ready for {len(splits_df)} series.")
print(f"   Train lengths: {splits_df['train_sc'].apply(len).describe().loc[['min','50%','max']].to_dict()}")

In [ ]:
# ============================================================
# CELL 8 (UPDATED): Apply Data Scarcity Regimes
# ============================================================
def get_regime_train(train_sc, regime):
    """Slice training data to a scarcity regime (all absolute now)."""
    return train_sc[:min(regime, len(train_sc))]

# Sanity check
ex = splits_df.iloc[0]["train_sc"]
for name, r in REGIMES.items():
    sl = get_regime_train(ex, r)
    print(f"  {name:10s}: {len(sl):3d} timesteps")

## Train Baselines

In [ ]:
# ============================================================
# CELL 9: Naive + Croston + ADIDA Baselines (statsforecast)
# ============================================================
from statsforecast import StatsForecast
from statsforecast.models import (
    CrostonOptimized,
    ADIDA,
    SeasonalNaive,
    Naive,
)

def build_sf_df(splits_df, regime_key):
    """Build long-format DataFrame for statsforecast."""
    rows = []
    for _, row in splits_df.iterrows():
        tr = get_regime_train(row["train_sc"], REGIMES[regime_key])
        for t, val in enumerate(tr):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds": pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y": float(val),
            })
    return pd.DataFrame(rows)

# Use 'full' regime first
sf_train = build_sf_df(splits_df, "full")
print(f"StatsForecast DF shape: {sf_train.shape}")
print(sf_train.groupby("unique_id").size().describe().loc[["min","50%","max"]])

sf = StatsForecast(
    models=[CrostonOptimized(), ADIDA(), SeasonalNaive(season_length=12), Naive()],
    freq="MS",      # monthly start
    n_jobs=-1,
)
sf.fit(sf_train)
forecasts = sf.predict(h=TEST_MONTHS)
print(f"\n✅ Baseline forecasts generated: {forecasts.shape}")
forecasts.head()

## MASE metric on baselines

In [ ]:
# ============================================================
# CELL 10: MASE Computation — First Checkpoint
# ============================================================
def mase(y_true, y_pred, y_train, seasonality=1):
    """Mean Absolute Scaled Error against seasonal naive."""
    naive_err = np.mean(np.abs(
        y_train[seasonality:] - y_train[:-seasonality]
    ))
    if naive_err < 1e-8:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / naive_err

model_cols = ["CrostonOptimized", "ADIDA", "SeasonalNaive", "Naive"]
results = []

for _, row in splits_df.iterrows():
    uid = str(row["series_id"])
    fc_row = forecasts[forecasts["unique_id"] == uid]
    if fc_row.empty:
        continue

    y_true = row["test_sc"]
    y_train = get_regime_train(row["train_sc"], REGIMES["full"])

    for col in model_cols:
        if col not in fc_row.columns:
            continue
        y_pred = fc_row[col].values[:TEST_MONTHS]
        if len(y_pred) != TEST_MONTHS:
            continue
        m = mase(y_true, y_pred, y_train, seasonality=12)
        results.append({"series_id": uid, "model": col, "mase": m})

results_df = pd.DataFrame(results)
summary = (results_df
           .groupby("model")["mase"]
           .agg(["median", "mean", "std"])
           .round(4)
           .sort_values("median"))

print("=" * 45)
print("📊 CHECKPOINT 1 — Real-Only Baselines (Full Regime)")
print("=" * 45)
print(summary.to_string())

## Neural forecast Data frame changes


In [ ]:
# ============================================================
# CELL 12: Build NeuralForecast-compatible long-format DataFrame
# ============================================================
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

def build_nf_df(splits_df, regime_key):
    """Long-format DataFrame for neuralforecast (full series per regime)."""
    rows = []
    for _, row in splits_df.iterrows():
        tr = get_regime_train(row["train_sc"], REGIMES[regime_key])
        n  = len(tr)
        for t, val in enumerate(tr):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })
    df = pd.DataFrame(rows)
    print(f"NeuralForecast DF — regime='{regime_key}': "
          f"{df['unique_id'].nunique()} series, "
          f"{len(df)} rows, "
          f"avg len={len(df)/df['unique_id'].nunique():.1f}")
    return df

nf_train_full = build_nf_df(splits_df, "full")
nf_train_full.head()

## Train PatchTST - Real data

### Train

In [ ]:
# ============================================================
# CELL 13 (FIXED): Train PatchTST — Correct API Parameters
# ============================================================

# Step 1: Drop NaN/inf
before = len(nf_train_full)
nf_train_full = nf_train_full.replace([np.inf, -np.inf], np.nan).dropna(subset=["y"])
after = len(nf_train_full)
print(f"Dropped {before - after} rows with NaN/inf in 'y'")

# Step 2: Drop short series
MIN_ROWS = 24
series_lengths = nf_train_full.groupby("unique_id")["y"].count()
valid_ids = series_lengths[series_lengths >= MIN_ROWS].index
nf_train_full = nf_train_full[nf_train_full["unique_id"].isin(valid_ids)]
print(f"Series remaining: {nf_train_full['unique_id'].nunique()}")

# Step 3: Check actual PatchTST signature
import inspect
from neuralforecast.models import PatchTST
print("\nPatchTST valid params:")
print([p for p in inspect.signature(PatchTST.__init__).parameters.keys()])

In [ ]:
# ============================================================
# CELL 13b (FIXED): Train PatchTST — no early stopping issue
# ============================================================
model = PatchTST(
    h                         = TEST_MONTHS,  # 12
    input_size                = 24,
    patch_len                 = 6,
    stride                    = 3,
    n_heads                   = 4,
    encoder_layers            = 3,
    hidden_size               = 128,
    linear_hidden_size        = 256,
    dropout                   = 0.1,
    max_steps                 = 500,
    batch_size                = 32,
    scaler_type               = "standard",
    random_seed               = SEED,
    early_stop_patience_steps = -1,           # -1 disables early stopping
)

nf = NeuralForecast(models=[model], freq="MS")
nf.fit(nf_train_full, val_size=TEST_MONTHS)   # reserve last 12 steps per series as val
print("✅ PatchTST training complete.")

### Predict

In [ ]:
# ============================================================
# CELL 14: Generate PatchTST Forecasts
# ============================================================
nf_preds = nf.predict()
nf_preds = nf_preds.reset_index()
print(f"Forecast shape: {nf_preds.shape}")
nf_preds.head()

### COmparison with baselines

In [ ]:
# ============================================================
# CELL 15: Compute MASE for PatchTST & Compare with Baselines
# ============================================================
patchtst_results = []

for _, row in splits_df.iterrows():
    uid     = str(row["series_id"])
    fc_row  = nf_preds[nf_preds["unique_id"] == uid]
    if fc_row.empty:
        continue

    y_true  = row["test_sc"]
    y_train = get_regime_train(row["train_sc"], REGIMES["full"])
    y_pred  = fc_row["PatchTST"].values[:TEST_MONTHS]

    if len(y_pred) != TEST_MONTHS:
        continue

    m = mase(y_true, y_pred, y_train, seasonality=1)  # seasonality=1 (not 12 — too few obs)
    patchtst_results.append({"series_id": uid, "model": "PatchTST", "mase": m})

patchtst_df = pd.DataFrame(patchtst_results)

# Combine with statsforecast results
all_results = pd.concat([results_df, patchtst_df], ignore_index=True)

summary_all = (all_results
               .groupby("model")["mase"]
               .agg(["median", "mean", "std"])
               .round(4)
               .sort_values("median"))

print("=" * 50)
print("📊 CHECKPOINT 1b — All Real-Only Baselines (Full Regime)")
print("=" * 50)
print(summary_all.to_string())

### Save checkpoints

In [ ]:
# ============================================================
# CELL 16: Save Checkpoint 1 Results
# ============================================================
import os, json
from datetime import datetime

os.makedirs("/content/results", exist_ok=True)

# Save MASE results to CSV
all_results.to_csv("/content/results/checkpoint1_real_only_full_regime.csv", index=False)

# Save summary as JSON for quick reference
summary_dict = summary_all[["median", "mean", "std"]].to_dict()
summary_dict["meta"] = {
    "regime"      : "full",
    "train_steps" : 39,
    "test_steps"  : TEST_MONTHS,
    "n_series"    : len(splits_df),
    "timestamp"   : datetime.now().isoformat(),
}
with open("/content/results/checkpoint1_summary.json", "w") as f:
    json.dump(summary_dict, f, indent=2)

print("✅ Checkpoint 1 saved.")
print(f"   → /content/results/checkpoint1_real_only_full_regime.csv")
print(f"   → /content/results/checkpoint1_summary.json")

### Regime decomposition using raw space

In [ ]:
# ============================================================
# CELL 17 (FIXED): Regime Decomposition in Raw Space
# ============================================================

# Step 1: Verify what test_sc actually looks like
sample = splits_df.iloc[0]
print("test_sc sample :", sample["test_sc"][:12])
print("test_raw sample:", sample["test_raw"][:12])
print("Any exact zeros in test_sc?", (splits_df["test_sc"].apply(lambda x: (x == 0).any())).sum())
print("Any exact zeros in test_raw?", (splits_df["test_raw"].apply(lambda x: (x == 0).any())).sum())

# Step 2: Compute threshold from RAW test values (not scaled)
all_test_nonzero_raw = np.concatenate([
    row["test_raw"][row["test_raw"] > 0]
    for _, row in splits_df.iterrows()
    if (row["test_raw"] > 0).any()
])
print(f"\nNon-zero raw test values count: {len(all_test_nonzero_raw)}")
print(f"Sample: {all_test_nonzero_raw[:10]}")

HIGH_THRESH_RAW = np.percentile(all_test_nonzero_raw, 75)
print(f"\nHigh-demand threshold (75th pct of non-zero raw test): {HIGH_THRESH_RAW:.4f}")

In [ ]:
# ============================================================
# CELL 17b: Regime Decomposition using raw space for masking
# ============================================================
def mase_decomposed_raw(y_true_sc, y_pred_sc, y_true_raw, y_train_sc,
                        threshold_raw, seasonality=1):
    """
    Masks are determined from RAW values (exact zero detection).
    MASE is computed in SCALED space (consistent with global metrics).
    """
    naive_err = np.mean(np.abs(
        y_train_sc[seasonality:] - y_train_sc[:-seasonality]
    ))
    if naive_err < 1e-8:
        return {"zero": np.nan, "low": np.nan, "high": np.nan}

    masks = {
        "zero": y_true_raw == 0,
        "low" : (y_true_raw > 0) & (y_true_raw < threshold_raw),
        "high": y_true_raw >= threshold_raw,
    }

    out = {}
    for regime, mask in masks.items():
        if mask.sum() == 0:
            out[regime] = np.nan
        else:
            out[regime] = float(
                np.mean(np.abs(y_true_sc[mask] - y_pred_sc[mask])) / naive_err
            )
    return out

# Run decomposition
decomp_rows = []

for _, row in splits_df.iterrows():
    uid       = str(row["series_id"])
    y_true_sc = row["test_sc"]
    y_true_raw= row["test_raw"]
    y_train_sc= get_regime_train(row["train_sc"], REGIMES["full"])

    # Stats models
    fc_stats = forecasts[forecasts["unique_id"] == uid]
    for col in ["Naive", "ADIDA", "SeasonalNaive", "CrostonOptimized"]:
        if fc_stats.empty or col not in fc_stats.columns:
            continue
        y_pred = fc_stats[col].values[:TEST_MONTHS]
        if len(y_pred) != TEST_MONTHS:
            continue
        d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                                y_train_sc, HIGH_THRESH_RAW)
        decomp_rows.append({"series_id": uid, "model": col, **d})

    # PatchTST
    fc_pt = nf_preds[nf_preds["unique_id"] == uid]
    if not fc_pt.empty:
        y_pred = fc_pt["PatchTST"].values[:TEST_MONTHS]
        if len(y_pred) == TEST_MONTHS:
            d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                                    y_train_sc, HIGH_THRESH_RAW)
            decomp_rows.append({"series_id": uid, "model": "PatchTST", **d})

decomp_df = pd.DataFrame(decomp_rows)

decomp_summary = (decomp_df
                  .groupby("model")[["zero", "low", "high"]]
                  .median()
                  .round(4)
                  .sort_values("high"))

print("=" * 55)
print("📊 CHECKPOINT 1c — Regime Decomposition (Median MASE)")
print("=" * 55)
print(decomp_summary.to_string())

# Show regime coverage so we know how populated each bucket is
print("\n📊 Regime coverage (% of test timesteps per bucket):")
for regime in ["zero", "low", "high"]:
    mask_counts = np.array([
        (row["test_raw"] == 0).sum() if regime == "zero"
        else ((row["test_raw"] > 0) & (row["test_raw"] < HIGH_THRESH_RAW)).sum() if regime == "low"
        else (row["test_raw"] >= HIGH_THRESH_RAW).sum()
        for _, row in splits_df.iterrows()
    ])
    total = 12 * len(splits_df)
    print(f"  {regime:4s}: {mask_counts.sum():6,} / {total:,} "
          f"({100*mask_counts.sum()/total:.1f}%)")

#### save checkpoint

In [ ]:
# ============================================================
# CELL 18: Save Checkpoint 1c + Lock Baseline for Comparison
# ============================================================
decomp_df.to_csv("/content/results/checkpoint1c_regime_decomposition.csv", index=False)
decomp_summary.to_csv("/content/results/checkpoint1c_summary.csv")

# summary_all has 'model' as index from groupby — use directly
BASELINE = {
    "global": summary_all["median"].to_dict(),   # model is already the index
    "regime": decomp_summary.to_dict(),           # zero/low/high
}

print("✅ Checkpoint 1c saved. Baselines locked.")
print("\n📌 Target gaps for augmentation to close (PatchTST):")
print(f"   Global MASE : {BASELINE['global']['PatchTST']:.4f}  → beat Naive @ 0.6385")
print(f"   High MASE   : {decomp_summary.loc['PatchTST','high']:.4f}  → beat Croston @ 0.7519")
print(f"   Zero MASE   : {decomp_summary.loc['PatchTST','zero']:.4f}  → maintain < 0.5")

## Jittering augmentation

### train baselines with Jitter

In [ ]:
# ============================================================
# CELL 19: Week 2 — Jittering Augmentation (Tier 1 Baseline)
# Expected to FAIL — smooths spikes, hurts high-demand regime
# ============================================================
def jitter_series(series, noise_std=0.1):
    """Add Gaussian noise: x' = x + N(0, noise_std * sigma)."""
    sigma = series.std()
    noise = np.random.normal(0, noise_std * sigma, size=len(series))
    return series + noise

def augment_jitter(splits_df, regime_key, ratio=1.0, noise_std=0.1):
    """
    Build augmented training DataFrame.
    ratio = synthetic:real (1.0 = equal amounts of synthetic and real).
    """
    rows = []
    syn_id_offset = len(splits_df) * 10  # avoid ID collision

    for idx, row in splits_df.iterrows():
        tr = get_regime_train(row["train_sc"], REGIMES[regime_key])
        n_real = len(tr)
        n_synth = max(1, int(n_real * ratio))

        # Real rows
        for t, val in enumerate(tr):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })

        # Synthetic rows (new virtual series)
        for s in range(int(ratio)):
            syn_series = jitter_series(tr, noise_std=noise_std)
            syn_id = f"syn_{row['series_id']}_{s}"
            for t, val in enumerate(syn_series):
                rows.append({
                    "unique_id": syn_id,
                    "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                    "y"        : float(val),
                })

    df = pd.DataFrame(rows)
    # Drop NaN/inf
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["y"])
    # Drop short series
    lens = df.groupby("unique_id")["y"].count()
    df   = df[df["unique_id"].isin(lens[lens >= 24].index)]
    print(f"Jitter augmented DF — real+synth series: {df['unique_id'].nunique()}, "
          f"rows: {len(df)}")
    return df

# Build at R=1.0x first
nf_train_jitter = augment_jitter(splits_df, "full", ratio=1.0, noise_std=0.1)

### Train PatchTST on Jitter

In [ ]:
# ============================================================
# CELL 20: Train PatchTST on Jittered Data
# ============================================================
model_jitter = PatchTST(
    h                         = TEST_MONTHS,
    input_size                = 24,
    patch_len                 = 6,
    stride                    = 3,
    n_heads                   = 4,
    encoder_layers            = 3,
    hidden_size               = 128,
    linear_hidden_size        = 256,
    dropout                   = 0.1,
    max_steps                 = 500,
    batch_size                = 32,
    scaler_type               = "standard",
    random_seed               = SEED,
    early_stop_patience_steps = -1,
)

nf_jitter = NeuralForecast(models=[model_jitter], freq="MS")
nf_jitter.fit(nf_train_jitter, val_size=TEST_MONTHS)
print("✅ PatchTST + Jitter training complete.")

### Evaluate Jitter vs Real only baseline

In [ ]:
# ============================================================
# CELL 21: Evaluate Jitter vs Real-Only Baseline
# ============================================================
# Predict only on REAL series (not synthetic ones)
real_ids     = set(str(r["series_id"]) for _, r in splits_df.iterrows())
futr_df      = nf_train_full[nf_train_full["unique_id"].isin(real_ids)]
preds_jitter = nf_jitter.predict(futr_df)
preds_jitter = preds_jitter.reset_index()

# MASE + decomposition
jitter_rows  = []
jitter_decomp= []

for _, row in splits_df.iterrows():
    uid       = str(row["series_id"])
    fc        = preds_jitter[preds_jitter["unique_id"] == uid]
    if fc.empty:
        continue
    y_true_sc = row["test_sc"]
    y_true_raw= row["test_raw"]
    y_train_sc= get_regime_train(row["train_sc"], REGIMES["full"])
    y_pred    = fc["PatchTST"].values[:TEST_MONTHS]
    if len(y_pred) != TEST_MONTHS:
        continue

    m = mase(y_true_sc, y_pred, y_train_sc, seasonality=1)
    jitter_rows.append({"series_id": uid, "model": "PatchTST+Jitter", "mase": m})

    d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                            y_train_sc, HIGH_THRESH_RAW)
    jitter_decomp.append({"series_id": uid, "model": "PatchTST+Jitter", **d})

jitter_df       = pd.DataFrame(jitter_rows)
jitter_decomp_df= pd.DataFrame(jitter_decomp)

print("=" * 55)
print("📊 CHECKPOINT 2 — Jitter vs Real-Only (PatchTST, Full Regime)")
print("=" * 55)
compare = pd.DataFrame({
    "Real-Only" : [0.9376, 0.4079, 1.9427, 2.8475],
    "Jitter R=1": [
        jitter_df["mase"].median(),
        jitter_decomp_df["zero"].median(),
        jitter_decomp_df["low"].median(),
        jitter_decomp_df["high"].median(),
    ]
}, index=["Global", "Zero", "Low", "High"])
print(compare.round(4))
print("\n📌 Delta (Jitter - Real-Only), negative = improvement:")
print((compare["Jitter R=1"] - compare["Real-Only"]).round(4))

### save Jitter results

In [ ]:
# ============================================================
# CELL 22: Save Checkpoint 2 + Poisson Bootstrap Augmentation
# ============================================================
jitter_df.to_csv("/content/results/checkpoint2_jitter_global.csv", index=False)
jitter_decomp_df.to_csv("/content/results/checkpoint2_jitter_decomp.csv", index=False)
print("✅ Checkpoint 2 saved.")

## Poisson augmentation


### Train


In [ ]:
# ============================================================
# CELL 23: Poisson Bootstrap Augmentation (Tier 2)
# Structure-preserving: resamples inter-arrival times and
# spike magnitudes separately — respects zero-inflation pattern
# ============================================================
def poisson_bootstrap_series(series, random_state=None):
    """
    Poisson Bootstrap for intermittent demand:
    1. Extract non-zero values (spike magnitudes)
    2. Extract inter-arrival intervals (gaps between spikes)
    3. Resample both independently with replacement
    4. Reconstruct a new series of the same length
    """
    rng = np.random.default_rng(random_state)
    n   = len(series)

    # Find spike positions and magnitudes
    spike_idx  = np.where(series != 0)[0]
    spike_vals = series[spike_idx]

    if len(spike_vals) == 0:
        # All-zero series — return as-is
        return series.copy()

    if len(spike_vals) == 1:
        # Single spike — resample magnitude only, keep position
        new_series = np.zeros(n, dtype=np.float32)
        new_series[spike_idx[0]] = float(rng.choice(spike_vals))
        return new_series

    # Inter-arrival intervals
    intervals = np.diff(spike_idx)  # gaps between consecutive spikes
    lead_time = spike_idx[0]        # gap before first spike

    # Resample magnitudes and intervals with replacement
    new_mags      = rng.choice(spike_vals, size=len(spike_vals), replace=True)
    new_intervals = rng.choice(intervals,  size=len(intervals),  replace=True)
    new_lead      = int(rng.choice(
        np.append(intervals, lead_time), size=1
    ))

    # Reconstruct series
    new_series  = np.zeros(n, dtype=np.float32)
    pos         = new_lead
    new_series[min(pos, n-1)] = new_mags[0]

    for i, gap in enumerate(new_intervals):
        pos += gap
        if pos >= n:
            break
        new_series[pos] = new_mags[i + 1] if (i + 1) < len(new_mags) else 0.0

    return new_series


def augment_poisson_bootstrap(splits_df, regime_key, ratio=1.0):
    """Build augmented DataFrame using Poisson Bootstrap."""
    rows = []
    for idx, row in splits_df.iterrows():
        tr     = get_regime_train(row["train_sc"], REGIMES[regime_key])
        n_real = len(tr)

        # Real rows
        for t, val in enumerate(tr):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })

        # Synthetic rows
        for s in range(int(ratio)):
            syn_series = poisson_bootstrap_series(tr, random_state=SEED + idx + s)
            syn_id     = f"synpb_{row['series_id']}_{s}"
            for t, val in enumerate(syn_series):
                rows.append({
                    "unique_id": syn_id,
                    "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                    "y"        : float(val),
                })

    df = pd.DataFrame(rows)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["y"])
    lens = df.groupby("unique_id")["y"].count()
    df   = df[df["unique_id"].isin(lens[lens >= 24].index)]
    print(f"Poisson Bootstrap DF — series: {df['unique_id'].nunique()}, "
          f"rows: {len(df)}")
    return df

nf_train_pb = augment_poisson_bootstrap(splits_df, "full", ratio=1.0)

In [ ]:
# ============================================================
# CELL 24: Train PatchTST on Poisson Bootstrap Data
# ============================================================
model_pb = PatchTST(
    h                         = TEST_MONTHS,
    input_size                = 24,
    patch_len                 = 6,
    stride                    = 3,
    n_heads                   = 4,
    encoder_layers            = 3,
    hidden_size               = 128,
    linear_hidden_size        = 256,
    dropout                   = 0.1,
    max_steps                 = 500,
    batch_size                = 32,
    scaler_type               = "standard",
    random_seed               = SEED,
    early_stop_patience_steps = -1,
)

nf_pb = NeuralForecast(models=[model_pb], freq="MS")
nf_pb.fit(nf_train_pb, val_size=TEST_MONTHS)
print("✅ PatchTST + Poisson Bootstrap training complete.")

### Evaluate with baselines

In [ ]:
# ============================================================
# CELL 25: Evaluate Poisson Bootstrap vs Baselines
# ============================================================
preds_pb = nf_pb.predict(
    nf_train_full[nf_train_full["unique_id"].isin(real_ids)]
).reset_index()

pb_rows   = []
pb_decomp = []

for _, row in splits_df.iterrows():
    uid        = str(row["series_id"])
    fc         = preds_pb[preds_pb["unique_id"] == uid]
    if fc.empty:
        continue
    y_true_sc  = row["test_sc"]
    y_true_raw = row["test_raw"]
    y_train_sc = get_regime_train(row["train_sc"], REGIMES["full"])
    y_pred     = fc["PatchTST"].values[:TEST_MONTHS]
    if len(y_pred) != TEST_MONTHS:
        continue

    m = mase(y_true_sc, y_pred, y_train_sc, seasonality=1)
    pb_rows.append({"series_id": uid, "model": "PatchTST+PoissonBS", "mase": m})

    d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                            y_train_sc, HIGH_THRESH_RAW)
    pb_decomp.append({"series_id": uid, "model": "PatchTST+PoissonBS", **d})

pb_df       = pd.DataFrame(pb_rows)
pb_decomp_df= pd.DataFrame(pb_decomp)

print("=" * 60)
print("📊 CHECKPOINT 3 — Augmentation Comparison (PatchTST, Full Regime)")
print("=" * 60)
compare2 = pd.DataFrame({
    "Real-Only" : [0.9376, 0.4079, 1.9427, 2.8475],
    "Jitter R=1": [0.9502, 0.4324, 1.9349, 2.8399],
    "PoissonBS R=1": [
        pb_df["mase"].median(),
        pb_decomp_df["zero"].median(),
        pb_decomp_df["low"].median(),
        pb_decomp_df["high"].median(),
    ]
}, index=["Global", "Zero", "Low", "High"])

print(compare2.round(4))
print("\n📌 Delta vs Real-Only (negative = improvement):")
for col in ["Jitter R=1", "PoissonBS R=1"]:
    print(f"\n  {col}:")
    print((compare2[col] - compare2["Real-Only"]).round(4).to_string())

### Poisson bootstrap in raw space

#### Train

In [ ]:
# ============================================================
# CELL 26: Poisson Bootstrap in RAW space (fix the root cause)
# Augment on train_raw → scale afterward
# ============================================================
def augment_poisson_bootstrap_raw(splits_df, regime_key, ratio=1.0):
    """
    Poisson Bootstrap operates on RAW values (not scaled).
    Synthetic series are then log(1+x) + Z-scored using the
    real training series stats — same pipeline as real data.
    """
    rows = []

    for idx, row in splits_df.iterrows():
        # Raw training slice
        tr_raw = row["train_raw"][:REGIMES[regime_key]]
        mu     = row["mu"]
        sigma  = row["sigma"]

        # Real rows (already scaled — use train_sc)
        tr_sc = get_regime_train(row["train_sc"], REGIMES[regime_key])
        for t, val in enumerate(tr_sc):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })

        # Synthetic: bootstrap in raw space, then scale
        for s in range(int(ratio)):
            syn_raw = poisson_bootstrap_series(
                tr_raw.astype(np.float32),
                random_state=SEED + idx + s
            )
            # Apply same log1p + Z-score as real pipeline
            syn_sc = (np.log1p(np.clip(syn_raw, 0, None)) - mu) / max(sigma, 1e-6)
            syn_id = f"synpbr_{row['series_id']}_{s}"
            for t, val in enumerate(syn_sc):
                rows.append({
                    "unique_id": syn_id,
                    "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                    "y"        : float(val),
                })

    df = pd.DataFrame(rows)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["y"])
    lens = df.groupby("unique_id")["y"].count()
    df   = df[df["unique_id"].isin(lens[lens >= 24].index)]
    print(f"PoissonBS-Raw DF — series: {df['unique_id'].nunique()}, rows: {len(df)}")
    return df

nf_train_pbr = augment_poisson_bootstrap_raw(splits_df, "full", ratio=1.0)

In [ ]:
# ============================================================
# CELL 27: Train PatchTST on Raw-Space Poisson Bootstrap
# ============================================================
model_pbr = PatchTST(
    h                         = TEST_MONTHS,
    input_size                = 24,
    patch_len                 = 6,
    stride                    = 3,
    n_heads                   = 4,
    encoder_layers            = 3,
    hidden_size               = 128,
    linear_hidden_size        = 256,
    dropout                   = 0.1,
    max_steps                 = 500,
    batch_size                = 32,
    scaler_type               = "standard",
    random_seed               = SEED,
    early_stop_patience_steps = -1,
)

nf_pbr = NeuralForecast(models=[model_pbr], freq="MS")
nf_pbr.fit(nf_train_pbr, val_size=TEST_MONTHS)
print("✅ PatchTST + PoissonBS-Raw training complete.")

#### Evaluate on baselines

In [ ]:
# ============================================================
# CELL 28: Evaluate Raw-Space PoissonBS
# ============================================================
preds_pbr = nf_pbr.predict(
    nf_train_full[nf_train_full["unique_id"].isin(real_ids)]
).reset_index()

pbr_rows   = []
pbr_decomp = []

for _, row in splits_df.iterrows():
    uid        = str(row["series_id"])
    fc         = preds_pbr[preds_pbr["unique_id"] == uid]
    if fc.empty:
        continue
    y_true_sc  = row["test_sc"]
    y_true_raw = row["test_raw"]
    y_train_sc = get_regime_train(row["train_sc"], REGIMES["full"])
    y_pred     = fc["PatchTST"].values[:TEST_MONTHS]
    if len(y_pred) != TEST_MONTHS:
        continue

    m = mase(y_true_sc, y_pred, y_train_sc, seasonality=1)
    pbr_rows.append({"series_id": uid, "model": "PatchTST+PoissonBS-Raw", "mase": m})

    d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                            y_train_sc, HIGH_THRESH_RAW)
    pbr_decomp.append({"series_id": uid, "model": "PatchTST+PoissonBS-Raw", **d})

pbr_df       = pd.DataFrame(pbr_rows)
pbr_decomp_df= pd.DataFrame(pbr_decomp)

print("=" * 65)
print("📊 CHECKPOINT 3b — Raw-Space PoissonBS vs Previous Methods")
print("=" * 65)
compare3 = pd.DataFrame({
    "Real-Only"     : [0.9376, 0.4079, 1.9427, 2.8475],
    "Jitter"        : [0.9502, 0.4324, 1.9349, 2.8399],
    "PoissonBS-Sc"  : [0.9510, 0.4131, 2.0155, 2.9122],
    "PoissonBS-Raw" : [
        pbr_df["mase"].median(),
        pbr_decomp_df["zero"].median(),
        pbr_decomp_df["low"].median(),
        pbr_decomp_df["high"].median(),
    ]
}, index=["Global", "Zero", "Low", "High"])

print(compare3.round(4))
print("\n📌 Delta PoissonBS-Raw vs Real-Only (negative = improvement):")
print((compare3["PoissonBS-Raw"] - compare3["Real-Only"]).round(4).to_string())

### Magnitude-Only Bootstrap

#### Train

In [ ]:
# ============================================================
# CELL 29: Magnitude-Only Bootstrap (Constrained PoissonBS)
# Keep inter-arrival intervals fixed, resample spike magnitudes only
# Hypothesis: interval misplacement is causing Low/High degradation
# ============================================================
def magnitude_only_bootstrap(series, random_state=None):
    """
    Resample spike magnitudes only — intervals stay fixed.
    Preserves zero-inflation pattern exactly.
    """
    rng        = np.random.default_rng(random_state)
    new_series = series.copy().astype(np.float32)
    spike_idx  = np.where(series != 0)[0]
    spike_vals = series[spike_idx]

    if len(spike_vals) <= 1:
        return new_series  # nothing to resample

    # Resample magnitudes in place — positions unchanged
    new_mags = rng.choice(spike_vals, size=len(spike_vals), replace=True)
    new_series[spike_idx] = new_mags
    return new_series


def augment_magnitude_bootstrap(splits_df, regime_key, ratio=1.0):
    """Magnitude-only bootstrap in raw space, then re-scale."""
    rows = []
    for idx, row in splits_df.iterrows():
        tr_raw = row["train_raw"][:REGIMES[regime_key]]
        mu     = row["mu"]
        sigma  = row["sigma"]
        tr_sc  = get_regime_train(row["train_sc"], REGIMES[regime_key])

        # Real rows
        for t, val in enumerate(tr_sc):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })

        # Synthetic rows — magnitude resampled, positions fixed
        for s in range(int(ratio)):
            syn_raw = magnitude_only_bootstrap(
                tr_raw.astype(np.float32),
                random_state=SEED + idx + s
            )
            syn_sc = (np.log1p(np.clip(syn_raw, 0, None)) - mu) / max(sigma, 1e-6)
            syn_id = f"synmb_{row['series_id']}_{s}"
            for t, val in enumerate(syn_sc):
                rows.append({
                    "unique_id": syn_id,
                    "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                    "y"        : float(val),
                })

    df = pd.DataFrame(rows)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["y"])
    lens = df.groupby("unique_id")["y"].count()
    df   = df[df["unique_id"].isin(lens[lens >= 24].index)]
    print(f"Magnitude-Bootstrap DF — series: {df['unique_id'].nunique()}, "
          f"rows: {len(df)}")
    return df

nf_train_mb = augment_magnitude_bootstrap(splits_df, "full", ratio=1.0)

In [ ]:
# ============================================================
# CELL 30: Train PatchTST on Magnitude-Only Bootstrap
# ============================================================
model_mb = PatchTST(
    h                         = TEST_MONTHS,
    input_size                = 24,
    patch_len                 = 6,
    stride                    = 3,
    n_heads                   = 4,
    encoder_layers            = 3,
    hidden_size               = 128,
    linear_hidden_size        = 256,
    dropout                   = 0.1,
    max_steps                 = 500,
    batch_size                = 32,
    scaler_type               = "standard",
    random_seed               = SEED,
    early_stop_patience_steps = -1,
)

nf_mb = NeuralForecast(models=[model_mb], freq="MS")
nf_mb.fit(nf_train_mb, val_size=TEST_MONTHS)
print("✅ PatchTST + Magnitude Bootstrap training complete.")

#### Evaluate

In [ ]:
# ============================================================
# CELL 31: Evaluate Magnitude Bootstrap
# ============================================================
preds_mb = nf_mb.predict(
    nf_train_full[nf_train_full["unique_id"].isin(real_ids)]
).reset_index()

mb_rows   = []
mb_decomp = []

for _, row in splits_df.iterrows():
    uid        = str(row["series_id"])
    fc         = preds_mb[preds_mb["unique_id"] == uid]
    if fc.empty:
        continue
    y_true_sc  = row["test_sc"]
    y_true_raw = row["test_raw"]
    y_train_sc = get_regime_train(row["train_sc"], REGIMES["full"])
    y_pred     = fc["PatchTST"].values[:TEST_MONTHS]
    if len(y_pred) != TEST_MONTHS:
        continue

    m = mase(y_true_sc, y_pred, y_train_sc, seasonality=1)
    mb_rows.append({"series_id": uid, "model": "PatchTST+MagBS", "mase": m})

    d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                            y_train_sc, HIGH_THRESH_RAW)
    mb_decomp.append({"series_id": uid, "model": "PatchTST+MagBS", **d})

mb_df       = pd.DataFrame(mb_rows)
mb_decomp_df= pd.DataFrame(mb_decomp)

print("=" * 65)
print("📊 CHECKPOINT 3c — Magnitude Bootstrap vs All Previous")
print("=" * 65)
compare4 = pd.DataFrame({
    "Real-Only"    : [0.9376, 0.4079, 1.9427, 2.8475],
    "Jitter"       : [0.9502, 0.4324, 1.9349, 2.8399],
    "PoissonBS-Sc" : [0.9510, 0.4131, 2.0155, 2.9122],
    "PoissonBS-Raw": [0.9362, 0.3897, 1.9770, 2.8858],
    "MagBS"        : [
        mb_df["mase"].median(),
        mb_decomp_df["zero"].median(),
        mb_decomp_df["low"].median(),
        mb_decomp_df["high"].median(),
    ]
}, index=["Global", "Zero", "Low", "High"])

print(compare4.round(4))
print("\n📌 Delta MagBS vs Real-Only (negative = improvement):")
print((compare4["MagBS"] - compare4["Real-Only"]).round(4).to_string())

# Save
mb_df.to_csv("/content/results/checkpoint3c_magbs_global.csv", index=False)
mb_decomp_df.to_csv("/content/results/checkpoint3c_magbs_decomp.csv", index=False)

#### Save Checkpoints

In [ ]:
# ============================================================
# CELL 32: Save Tier 2 Summary + Lock Best Bootstrap Method
# ============================================================
tier2_summary = pd.DataFrame({
    "Real-Only"    : [0.9376, 0.4079, 1.9427, 2.8475],
    "Jitter"       : [0.9502, 0.4324, 1.9349, 2.8399],
    "PoissonBS-Sc" : [0.9510, 0.4131, 2.0155, 2.9122],
    "PoissonBS-Raw": [0.9362, 0.3897, 1.9770, 2.8858],
    "MagBS"        : [0.9402, 0.4048, 1.9551, 2.8831],
}, index=["Global", "Zero", "Low", "High"])

tier2_summary.to_csv("/content/results/tier2_bootstrap_summary.csv")
print("✅ Tier 2 locked. Best method: PoissonBS-Raw (Global: 0.9362, Zero: 0.3897)")
print("\n📌 Moving to Tier 3: TimeGAN — generative augmentation")
print("   Hypothesis: learned latent distribution captures spike")
print("   patterns that resampling cannot reproduce")

## TIME-GAN

### Data Preparation for TIME-GAN

In [ ]:
# ============================================================
# CELL 34 (FIXED): Prepare TimeGAN Input with NaN guards
# ============================================================
from sklearn.preprocessing import MinMaxScaler

def prepare_timegan_data(splits_df, regime_key, seq_len=24):
    X = []
    skipped = 0
    for _, row in splits_df.iterrows():
        tr_raw = row["train_raw"][:REGIMES[regime_key]]
        if len(tr_raw) < seq_len:
            skipped += 1
            continue

        vals = tr_raw[:seq_len].astype(np.float32)

        # Skip series with NaN/inf in raw values
        if np.isnan(vals).any() or np.isinf(vals).any():
            skipped += 1
            continue

        X.append(vals)

    X      = np.array(X)[:, :, np.newaxis]   # (N, seq_len, 1)
    print(f"Series kept: {len(X)}  skipped: {skipped}")
    print(f"Raw value range — min:{X.min():.2f}  max:{X.max():.2f}")

    # Check if max == min (would cause NaN in MinMaxScaler)
    val_min = X.min()
    val_max = X.max()
    if val_max - val_min < 1e-8:
        raise ValueError("All values identical — cannot MinMax scale")

    N, T, F = X.shape
    X_flat  = X.reshape(N * T, F)
    scaler  = MinMaxScaler(feature_range=(0.001, 0.999))  # avoid exact 0/1
    X_sc    = scaler.fit_transform(X_flat).reshape(N, T, F)

    # Final NaN check
    nan_count = np.isnan(X_sc).sum()
    inf_count = np.isinf(X_sc).sum()
    print(f"After scaling — min:{X_sc.min():.4f}  max:{X_sc.max():.4f}  "
          f"NaNs:{nan_count}  Infs:{inf_count}")

    if nan_count > 0 or inf_count > 0:
        # Find which positions have NaN
        bad_rows = np.where(np.isnan(X_sc).any(axis=(1,2)))[0]
        print(f"  Bad series indices: {bad_rows[:10]}")
        # Remove them
        good_mask = ~np.isnan(X_sc).any(axis=(1, 2))
        X_sc      = X_sc[good_mask]
        print(f"  Cleaned shape: {X_sc.shape}")

    return X_sc, scaler

X_tgan, tgan_scaler = prepare_timegan_data(splits_df, "full", seq_len=24)

# Confirm clean
print(f"\n✅ Final check — NaNs:{np.isnan(X_tgan).sum()}  "
      f"Infs:{np.isinf(X_tgan).sum()}  "
      f"shape:{X_tgan.shape}")

### Time-GAN Architecture

In [ ]:
# ============================================================
# CELL 35 (FIXED): TimeGAN Architecture — Anti-Collapse Version
# Key changes:
# - Remove sigmoid from embedder/generator outputs (use tanh)
# - Smaller hidden dim (32)
# - LayerNorm after GRU for stability
# ============================================================
SEQ_LEN    = 24
INPUT_DIM  = 1
HIDDEN_DIM = 32   # reduced from 64
NOISE_DIM  = 32
NUM_LAYERS = 2    # reduced from 3

class EmbedderNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.rnn  = nn.GRU(input_dim, hidden_dim,
                           num_layers=num_layers, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        h, _  = self.rnn(x)
        h     = self.norm(h)
        return torch.tanh(self.fc(h))   # tanh not sigmoid


class RecoveryNet(nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.rnn  = nn.GRU(hidden_dim, hidden_dim,
                           num_layers=num_layers, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Linear(hidden_dim, output_dim)

    def forward(self, h):
        out, _ = self.rnn(h)
        out    = self.norm(out)
        return torch.sigmoid(self.fc(out))  # sigmoid for [0,1] output


class GeneratorNet(nn.Module):
    def __init__(self, noise_dim, hidden_dim, num_layers):
        super().__init__()
        self.rnn  = nn.GRU(noise_dim, hidden_dim,
                           num_layers=num_layers, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, z):
        h, _  = self.rnn(z)
        h     = self.norm(h)
        return torch.tanh(self.fc(h))   # tanh not sigmoid


class SupervisorNet(nn.Module):
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.rnn  = nn.GRU(hidden_dim, hidden_dim,
                           num_layers=max(1, num_layers - 1),
                           batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, h):
        out, _ = self.rnn(h)
        out    = self.norm(out)
        return torch.tanh(self.fc(out))


class DiscriminatorNet(nn.Module):
    def __init__(self, hidden_dim, num_layers):
        super().__init__()
        self.rnn  = nn.GRU(hidden_dim, hidden_dim,
                           num_layers=num_layers, batch_first=True,
                           bidirectional=True)
        self.norm = nn.LayerNorm(hidden_dim * 2)
        self.fc   = nn.Linear(hidden_dim * 2, 1)

    def forward(self, h):
        out, _ = self.rnn(h)
        out    = self.norm(out)
        return self.fc(out)


embedder      = EmbedderNet(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
recovery      = RecoveryNet(HIDDEN_DIM, INPUT_DIM, NUM_LAYERS).to(DEVICE)
generator     = GeneratorNet(NOISE_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
supervisor    = SupervisorNet(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
discriminator = DiscriminatorNet(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)

print("✅ Revised TimeGAN architecture initialised")
print(f"   HIDDEN_DIM={HIDDEN_DIM}, NUM_LAYERS={NUM_LAYERS}")
print(f"   Embedder params     : {sum(p.numel() for p in embedder.parameters()):,}")
print(f"   Generator params    : {sum(p.numel() for p in generator.parameters()):,}")
print(f"   Discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

### Train

In [ ]:
# ============================================================
# CELL 36 (FIXED): TimeGAN Training with NaN guards
# ============================================================
BATCH_SIZE  = 128
EPOCHS_AE   = 100
EPOCHS_SUP  = 100
EPOCHS_JOINT= 100
GAMMA       = 1.0
LR          = 1e-4   # lowered from 1e-3 — main fix for NaN

# Verify input has no NaN/inf before training
assert not np.isnan(X_tgan).any(), "NaN in X_tgan!"
assert not np.isinf(X_tgan).any(), "Inf in X_tgan!"
assert X_tgan.min() >= 0.0 and X_tgan.max() <= 1.0, "Not in [0,1]!"
print(f"✅ Input verified — shape:{X_tgan.shape}  "
      f"min:{X_tgan.min():.4f}  max:{X_tgan.max():.4f}")

# Reinitialise models cleanly
embedder      = EmbedderNet(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
recovery      = RecoveryNet(HIDDEN_DIM, INPUT_DIM, NUM_LAYERS).to(DEVICE)
generator     = GeneratorNet(NOISE_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
supervisor    = SupervisorNet(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
discriminator = DiscriminatorNet(HIDDEN_DIM, NUM_LAYERS).to(DEVICE)

X_tensor = torch.FloatTensor(X_tgan).to(DEVICE)
loader   = DataLoader(TensorDataset(X_tensor),
                      batch_size=BATCH_SIZE, shuffle=True,
                      drop_last=True)   # avoid single-sample batches

mse_loss = nn.MSELoss()
bce_loss = nn.BCEWithLogitsLoss()

opt_embed = optim.Adam(list(embedder.parameters()) +
                       list(recovery.parameters()), lr=LR)
opt_sup   = optim.Adam(list(generator.parameters()) +
                       list(supervisor.parameters()), lr=LR)
opt_gen   = optim.Adam(list(generator.parameters()) +
                       list(supervisor.parameters()) +
                       list(embedder.parameters()) +
                       list(recovery.parameters()), lr=LR)
opt_disc  = optim.Adam(discriminator.parameters(), lr=LR)

CLIP = 1.0  # gradient clipping norm

def sample_noise(n, seq_len, noise_dim):
    return torch.FloatTensor(n, seq_len, noise_dim).uniform_(0, 1).to(DEVICE)

# ── Phase 1: Autoencoder ──────────────────────────────────
print("Phase 1: Autoencoder pretraining...")
for epoch in range(EPOCHS_AE):
    epoch_loss = []
    for (x_real,) in loader:
        opt_embed.zero_grad()
        h     = embedder(x_real)
        x_hat = recovery(h)
        loss  = mse_loss(x_hat, x_real)

        if torch.isnan(loss):
            print(f"  ⚠️  NaN loss at epoch {epoch+1} — skipping batch")
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(
            list(embedder.parameters()) + list(recovery.parameters()), CLIP
        )
        opt_embed.step()
        epoch_loss.append(loss.item())

    if (epoch + 1) % 20 == 0:
        avg = np.mean(epoch_loss) if epoch_loss else float("nan")
        print(f"  AE epoch {epoch+1:3d}/{EPOCHS_AE}  recon_loss={avg:.6f}")

# ── Phase 2: Supervisor ───────────────────────────────────
print("\nPhase 2: Supervisor pretraining...")
for epoch in range(EPOCHS_SUP):
    epoch_loss = []
    for (x_real,) in loader:
        opt_sup.zero_grad()
        with torch.no_grad():
            h = embedder(x_real)
        z     = sample_noise(x_real.size(0), SEQ_LEN, NOISE_DIM)
        e_hat = generator(z)
        h_hat = supervisor(e_hat)
        loss  = mse_loss(h_hat[:, :-1, :], h[:, 1:, :])

        if torch.isnan(loss):
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(
            list(generator.parameters()) + list(supervisor.parameters()), CLIP
        )
        opt_sup.step()
        epoch_loss.append(loss.item())

    if (epoch + 1) % 20 == 0:
        avg = np.mean(epoch_loss) if epoch_loss else float("nan")
        print(f"  Sup epoch {epoch+1:3d}/{EPOCHS_SUP}  sup_loss={avg:.6f}")

# ── Phase 3: Joint Training ───────────────────────────────
print("\nPhase 3: Joint training...")
for epoch in range(EPOCHS_JOINT):
    d_losses, g_losses = [], []
    for (x_real,) in loader:
        # ── Discriminator ──
        opt_disc.zero_grad()
        with torch.no_grad():
            h         = embedder(x_real)
            z         = sample_noise(x_real.size(0), SEQ_LEN, NOISE_DIM)
            e_hat     = generator(z)
            h_hat_sup = supervisor(e_hat)

        y_real  = discriminator(h)
        y_fake  = discriminator(h_hat_sup)
        y_fake2 = discriminator(e_hat)

        d_loss  = (bce_loss(y_real,  torch.ones_like(y_real))   +
                   bce_loss(y_fake,  torch.zeros_like(y_fake))  +
                   bce_loss(y_fake2, torch.zeros_like(y_fake2))) / 3

        if not torch.isnan(d_loss):
            d_loss.backward()
            nn.utils.clip_grad_norm_(discriminator.parameters(), CLIP)
            opt_disc.step()
            d_losses.append(d_loss.item())

        # ── Generator ──
        opt_gen.zero_grad()
        z         = sample_noise(x_real.size(0), SEQ_LEN, NOISE_DIM)
        e_hat     = generator(z)
        h_hat_sup = supervisor(e_hat)
        x_hat     = recovery(h_hat_sup)
        h         = embedder(x_real)

        g_loss_u  = bce_loss(discriminator(h_hat_sup),
                             torch.ones_like(discriminator(h_hat_sup)))
        g_loss_u2 = bce_loss(discriminator(e_hat),
                             torch.ones_like(discriminator(e_hat)))
        g_loss_s  = mse_loss(h_hat_sup[:, :-1, :], h[:, 1:, :].detach())
        g_loss_v1 = torch.mean(torch.abs(
                        x_hat.std(dim=0)  - x_real.std(dim=0)))
        g_loss_v2 = torch.mean(torch.abs(
                        x_hat.mean(dim=0) - x_real.mean(dim=0)))
        g_loss    = (g_loss_u + g_loss_u2 +
                     GAMMA * g_loss_s +
                     g_loss_v1 + g_loss_v2)

        if not torch.isnan(g_loss):
            g_loss.backward()
            nn.utils.clip_grad_norm_(
                list(generator.parameters()) +
                list(supervisor.parameters()) +
                list(embedder.parameters()) +
                list(recovery.parameters()), CLIP
            )
            opt_gen.step()
            g_losses.append(g_loss.item())

    if (epoch + 1) % 20 == 0:
        d_avg = np.mean(d_losses) if d_losses else float("nan")
        g_avg = np.mean(g_losses) if g_losses else float("nan")
        print(f"  Joint epoch {epoch+1:3d}/{EPOCHS_JOINT}  "
              f"d_loss={d_avg:.4f}  g_loss={g_avg:.4f}")

print("\n✅ TimeGAN training complete.")

### Generation and sanity checks

In [ ]:
# ============================================================
# CELL 37 (FIXED): Generate & Sanity Check TimeGAN Samples
# ============================================================
def generate_timegan_samples(n_samples):
    generator.eval()
    supervisor.eval()
    recovery.eval()
    with torch.no_grad():
        z     = sample_noise(n_samples, SEQ_LEN, NOISE_DIM)
        e_hat = generator(z)
        h_hat = supervisor(e_hat)
        x_hat = recovery(h_hat).cpu().numpy()
    N, T, F = x_hat.shape
    x_raw   = tgan_scaler.inverse_transform(
                  x_hat.reshape(N * T, F)
              ).reshape(N, T)
    return np.clip(x_raw, 0, None)

syn_samples = generate_timegan_samples(500)

# Compute real zero ratio from train_raw directly
real_zero_ratio = np.mean([
    (row["train_raw"] == 0).mean()
    for _, row in splits_df.iterrows()
])

ZERO_THRESH    = 0.5
syn_zero_ratio = (syn_samples < ZERO_THRESH).mean()

real_spikes = np.concatenate([
    row["train_raw"][row["train_raw"] > 0]
    for _, row in splits_df.iterrows()
    if (row["train_raw"] > 0).any()
])
syn_spikes = syn_samples[syn_samples >= ZERO_THRESH].flatten()

print("=" * 50)
print("📊 TimeGAN Sanity Check (n=500 generated series)")
print("=" * 50)
print(f"  Zero ratio  — Real: {real_zero_ratio:.3f}  |  "
      f"TimeGAN: {syn_zero_ratio:.3f}  "
      f"({'✅' if abs(syn_zero_ratio - real_zero_ratio) < 0.15 else '❌ too far'})")
print(f"  Spike mean  — Real: {real_spikes.mean():.3f}  |  "
      f"TimeGAN: {syn_spikes.mean():.3f}")
print(f"  Spike std   — Real: {real_spikes.std():.3f}  |  "
      f"TimeGAN: {syn_spikes.std():.3f}")
print(f"  Spike max   — Real: {real_spikes.max():.1f}   |  "
      f"TimeGAN: {syn_spikes.max():.3f}")
print(f"  Value range — [{syn_samples.min():.3f}, {syn_samples.max():.3f}]")

zero_gap = abs(syn_zero_ratio - real_zero_ratio)
if zero_gap < 0.15:
    print("\n✅ Zero ratio acceptable — proceed to Cell 38")
elif zero_gap < 0.25:
    print("\n⚠️  Zero ratio borderline — threshold values < 0.5 to zero")
else:
    print("\n❌ Zero ratio too far — increase GAMMA or epochs and retrain")

# Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
axes      = axes.flatten()
real_ex   = splits_df.sample(6, random_state=SEED)

for i, ax in enumerate(axes):
    real = real_ex.iloc[i]["train_raw"][:SEQ_LEN]
    syn  = syn_samples[i]
    ax.step(range(len(real)), real,
            where="mid", color="steelblue", label="Real", alpha=0.8)
    ax.step(range(len(syn)), syn,
            where="mid", color="darkorange", label="TimeGAN",
            alpha=0.8, linestyle="--")
    ax.set_title(f"Series {i+1}")
    ax.set_ylabel("Demand (raw)")
    ax.legend(fontsize=7)

plt.suptitle(f"TimeGAN — Zero ratio  Real:{real_zero_ratio:.2f} | "
             f"GAN:{syn_zero_ratio:.2f}", fontsize=12)
plt.tight_layout()
plt.show()

### Augmentation of DataFrame

In [ ]:
# ============================================================
# CELL 38: Build Augmented DataFrame from TimeGAN samples
# ============================================================
def augment_timegan(splits_df, regime_key, ratio=1.0, n_synth=None):
    """
    Generate TimeGAN synthetic series and merge with real data.
    n_synth: total synthetic series to generate (default: ratio * n_real)
    """
    n_real  = len(splits_df)
    n_synth = n_synth or int(n_real * ratio)

    # Generate all synthetic series at once
    syn_raw = generate_timegan_samples(n_synth)   # (n_synth, 24)

    rows = []
    # Real rows
    for _, row in splits_df.iterrows():
        tr_sc = get_regime_train(row["train_sc"], REGIMES[regime_key])
        for t, val in enumerate(tr_sc):
            rows.append({
                "unique_id": str(row["series_id"]),
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })

    # Synthetic rows — scale using global stats (no per-series stats available)
    global_mu    = np.log1p(np.concatenate(splits_df["train_raw"].values)).mean()
    global_sigma = np.log1p(np.concatenate(splits_df["train_raw"].values)).std()
    global_sigma = max(global_sigma, 1e-6)

    for s, syn in enumerate(syn_raw):
        syn_sc = (np.log1p(np.clip(syn, 0, None)) - global_mu) / global_sigma
        for t, val in enumerate(syn_sc):
            rows.append({
                "unique_id": f"syntgan_{s}",
                "ds"       : pd.Timestamp("2000-01") + pd.DateOffset(months=t),
                "y"        : float(val),
            })

    df   = pd.DataFrame(rows)
    df   = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["y"])
    lens = df.groupby("unique_id")["y"].count()
    df   = df[df["unique_id"].isin(lens[lens >= 24].index)]
    print(f"TimeGAN augmented DF — series: {df['unique_id'].nunique()}, "
          f"rows: {len(df)}")
    return df

nf_train_tgan = augment_timegan(splits_df, "full", ratio=1.0)

### Retrain with different checkpoint

In [ ]:
# ============================================================
# CELL 39 (RETRAIN): Force separate checkpoint with alias
# ============================================================
from neuralforecast.models import PatchTST
from neuralforecast import NeuralForecast

model_tgan = PatchTST(
    h                         = TEST_MONTHS,
    input_size                = 24,
    patch_len                 = 6,
    stride                    = 3,
    n_heads                   = 4,
    encoder_layers            = 3,
    hidden_size               = 128,
    linear_hidden_size        = 256,
    dropout                   = 0.1,
    max_steps                 = 500,
    batch_size                = 32,
    scaler_type               = "standard",
    random_seed               = SEED + 1,   # different seed forces new checkpoint
    early_stop_patience_steps = -1,
    alias                     = "PatchTST_TimeGAN",  # explicit unique name
)

# Train on TimeGAN augmented data
nf_tgan = NeuralForecast(
    models = [model_tgan],
    freq   = "MS",
)
nf_tgan.fit(nf_train_tgan, val_size=TEST_MONTHS)
print("✅ PatchTST+TimeGAN retrained with unique alias.")

# Predict immediately after fit in same cell — no caching issue
preds_tgan_v3 = nf_tgan.predict(nf_train_tgan).reset_index()
preds_tgan_v3 = preds_tgan_v3[
    preds_tgan_v3["unique_id"].isin(real_ids)
].copy()

print(f"Pred col name  : {[c for c in preds_tgan_v3.columns if 'PatchTST' in c]}")
print(f"Pred std       : {preds_tgan_v3.filter(like='PatchTST').values.std():.6f}")
print(f"Pred range     : [{preds_tgan_v3.filter(like='PatchTST').values.min():.4f}, "
      f"{preds_tgan_v3.filter(like='PatchTST').values.max():.4f}]")
print(f"Unique series  : {preds_tgan_v3['unique_id'].nunique()}")

#### Evaluation

In [ ]:
# ============================================================
# CELL 40 (FINAL): Evaluate with aliased model predictions
# ============================================================

# Column name changes with alias
pred_col = [c for c in preds_tgan_v3.columns
            if "PatchTST" in c and c != "unique_id" and c != "ds"][0]
print(f"Using prediction column: '{pred_col}'")

tgan_rows   = []
tgan_decomp = []

for _, row in splits_df.iterrows():
    uid        = str(row["series_id"])
    fc         = preds_tgan_v3[preds_tgan_v3["unique_id"] == uid]
    if fc.empty:
        continue
    y_true_sc  = row["test_sc"]
    y_true_raw = row["test_raw"]
    y_train_sc = get_regime_train(row["train_sc"], REGIMES["full"])
    y_pred     = fc[pred_col].values[:TEST_MONTHS]
    if len(y_pred) != TEST_MONTHS:
        continue

    m = mase(y_true_sc, y_pred, y_train_sc, seasonality=1)
    tgan_rows.append({"series_id": uid, "model": "PatchTST+TimeGAN", "mase": m})

    d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                            y_train_sc, HIGH_THRESH_RAW)
    tgan_decomp.append({"series_id": uid, "model": "PatchTST+TimeGAN", **d})

tgan_df        = pd.DataFrame(tgan_rows)
tgan_decomp_df = pd.DataFrame(tgan_decomp)

print(f"Evaluated series      : {len(tgan_rows)}")
print(f"MASE std across series: {tgan_df['mase'].std():.4f}")

print("\n" + "=" * 70)
print("📊 CHECKPOINT 4 (FINAL) — TimeGAN vs Key Methods")
print("=" * 70)
compare5 = pd.DataFrame({
    "Real-Only"    : [0.9376, 0.4079, 1.9427, 2.8475],
    "PoissonBS-Raw": [0.9362, 0.3897, 1.9770, 2.8858],
    "TimeGAN"      : [
        tgan_df["mase"].median(),
        tgan_decomp_df["zero"].median(),
        tgan_decomp_df["low"].median(),
        tgan_decomp_df["high"].median(),
    ]
}, index=["Global", "Zero", "Low", "High"])

print(compare5.round(4))
print("\n📌 Delta TimeGAN vs Real-Only (negative = improvement):")
print((compare5["TimeGAN"] - compare5["Real-Only"]).round(4).to_string())

tgan_df.to_csv("/content/results/checkpoint4_timegan_global.csv", index=False)
tgan_decomp_df.to_csv("/content/results/checkpoint4_timegan_decomp.csv", index=False)
compare5.to_csv("/content/results/checkpoint4_full_comparison.csv")
print("\n✅ Checkpoint 4 saved.")

### Summary of all methods

In [ ]:
# ============================================================
# CELL 41: Full Tier Summary — All Methods
# ============================================================
full_summary = pd.DataFrame({
    "Real-Only"    : [0.9376, 0.4079, 1.9427, 2.8475],
    "Jitter"       : [0.9502, 0.4324, 1.9349, 2.8399],
    "PoissonBS-Sc" : [0.9510, 0.4131, 2.0155, 2.9122],
    "PoissonBS-Raw": [0.9362, 0.3897, 1.9770, 2.8858],
    "MagBS"        : [0.9402, 0.4048, 1.9551, 2.8831],
    "TimeGAN"      : [0.9044, 0.3158, 1.9652, 2.8170],
}, index=["Global", "Zero", "Low", "High"])

print("=" * 75)
print("📊 FULL TIER SUMMARY — All Augmentation Methods (PatchTST, Full Regime)")
print("=" * 75)
print(full_summary.round(4).to_string())

print("\n📌 Delta vs Real-Only (negative = improvement):")
delta = full_summary.subtract(full_summary["Real-Only"], axis=0).drop(
    columns="Real-Only"
)
print(delta.round(4).to_string())

print("\n🏆 Best method per regime:")
for regime in ["Global", "Zero", "Low", "High"]:
    best_col = delta.loc[regime].idxmin()
    best_val = delta.loc[regime].min()
    print(f"  {regime:6s}: {best_col:15s}  Δ={best_val:+.4f}")

full_summary.to_csv("/content/results/full_tier_summary.csv")
delta.to_csv("/content/results/full_tier_delta.csv")
print("\n✅ Full tier summary saved.")

### TimeGAN ratio ablation

#### Train

In [ ]:
# ============================================================
# CELL 42 (FIXED v2): TimeGAN Ratio Ablation — reset lists
# ============================================================
RATIOS = [0.5, 1.0, 2.0, 3.0]
ratio_results = {}

for ratio in RATIOS:
    print(f"\n{'='*50}")
    print(f"R={ratio}...")

    nf_train_r = augment_timegan(splits_df, "full", ratio=ratio)
    alias      = f"PatchTST_TGAN_{int(ratio*10)}"

    model_r = PatchTST(
        h=TEST_MONTHS, input_size=24, patch_len=6, stride=3,
        n_heads=4, encoder_layers=3, hidden_size=128,
        linear_hidden_size=256, dropout=0.1, max_steps=500,
        batch_size=32, scaler_type="standard",
        random_seed=SEED+1, early_stop_patience_steps=-1,
        alias=alias,
    )
    nf_r = NeuralForecast(models=[model_r], freq="MS")
    nf_r.fit(nf_train_r, val_size=TEST_MONTHS)

    preds_r  = nf_r.predict(nf_train_r).reset_index()
    preds_r  = preds_r[preds_r["unique_id"].isin(real_ids)].copy()
    pred_col = [c for c in preds_r.columns
                if c not in ("unique_id", "ds")][0]
    print(f"  col='{pred_col}'  std={preds_r[pred_col].std():.4f}")

    # ── Reset lists every ratio iteration ──
    mase_list   = []   # reset here
    decomp_list = []   # reset here

    for _, row in splits_df.iterrows():
        uid        = str(row["series_id"])
        fc         = preds_r[preds_r["unique_id"] == uid]
        if fc.empty:
            continue
        y_true_sc  = row["test_sc"]
        y_true_raw = row["test_raw"]
        y_train_sc = get_regime_train(row["train_sc"], REGIMES["full"])
        y_pred     = fc[pred_col].values[:TEST_MONTHS]
        if len(y_pred) != TEST_MONTHS:
            continue

        m = mase(y_true_sc, y_pred, y_train_sc, seasonality=1)
        if not np.isnan(m):
            mase_list.append(m)

        d = mase_decomposed_raw(y_true_sc, y_pred, y_true_raw,
                                y_train_sc, HIGH_THRESH_RAW)
        decomp_list.append(d)

    decomp_df = pd.DataFrame(decomp_list)
    ratio_results[ratio] = {
        "Global": float(np.median(mase_list)) if mase_list else np.nan,
        "Zero"  : float(decomp_df["zero"].median()),
        "Low"   : float(decomp_df["low"].median()),
        "High"  : float(decomp_df["high"].median()),
    }
    print(f"  → Global:{ratio_results[ratio]['Global']:.4f}  "
          f"Zero:{ratio_results[ratio]['Zero']:.4f}  "
          f"Low:{ratio_results[ratio]['Low']:.4f}  "
          f"High:{ratio_results[ratio]['High']:.4f}  "
          f"(n={len(mase_list)})")

print("\n✅ Ratio ablation complete.")

#### Ablation summary

In [ ]:
# ============================================================
# CELL 43 (FIXED): Ratio Ablation Summary
# ============================================================
ratio_df = pd.DataFrame(ratio_results,
                        index=["Global","Zero","Low","High"]).T
ratio_df.index.name = "Ratio"

print("=" * 55)
print("📊 CHECKPOINT 5 — TimeGAN Ratio Ablation")
print("=" * 55)
print(ratio_df.round(4).to_string())

baseline = {"Global":0.9376, "Zero":0.4079, "Low":1.9427, "High":2.8475}
print("\n📌 Delta vs Real-Only:")
for ratio, row in ratio_df.iterrows():
    print(f"\n  R={ratio}:")
    for metric in ["Global","Zero","Low","High"]:
        delta = row[metric] - baseline[metric]
        flag  = "✅" if delta < 0 else "❌"
        print(f"    {metric:6s}: {delta:+.4f} {flag}")

print("\n🏆 Best ratio per regime:")
for metric in ["Global","Zero","Low","High"]:
    best_r = ratio_df[metric].idxmin()
    print(f"  {metric:6s}: R={best_r}  "
          f"({ratio_df.loc[best_r, metric]:.4f})")

ratio_df.to_csv("/content/results/checkpoint5_ratio_ablation.csv")
print("\n✅ Checkpoint 5 saved.")

### Regime Ablation

In [ ]:
# ============================================================
# CELL 44: Scarcity Regime Ablation
# Best method (TimeGAN R=1.0) vs Real-Only across all regimes
# ============================================================

regime_results = {}

for regime_key in ["extreme", "moderate", "full"]:
    print(f"\n{'='*50}")
    print(f"Regime: {regime_key} ({REGIMES[regime_key]} train steps)")

    # ── Real-Only ──
    nf_train_regime = build_nf_df(splits_df, regime_key)
    nf_train_regime = nf_train_regime.replace(
        [np.inf, -np.inf], np.nan).dropna(subset=["y"])
    lens = nf_train_regime.groupby("unique_id")["y"].count()
    nf_train_regime = nf_train_regime[
        nf_train_regime["unique_id"].isin(lens[lens >= 6].index)]

    model_real = PatchTST(
        h=TEST_MONTHS, input_size=min(12, REGIMES[regime_key]),
        patch_len=3, stride=1,
        n_heads=4, encoder_layers=3, hidden_size=128,
        linear_hidden_size=256, dropout=0.1, max_steps=500,
        batch_size=32, scaler_type="standard",
        random_seed=SEED, early_stop_patience_steps=-1,
        alias=f"PatchTST_Real_{regime_key}",
    )
    nf_real = NeuralForecast(models=[model_real], freq="MS")
    nf_real.fit(nf_train_regime, val_size=min(6, REGIMES[regime_key]//2))

    preds_real  = nf_real.predict(nf_train_regime).reset_index()
    preds_real  = preds_real[preds_real["unique_id"].isin(real_ids)]
    pred_col_r  = [c for c in preds_real.columns
                   if c not in ("unique_id","ds")][0]

    # ── TimeGAN Augmented ──
    nf_train_tgan_r = augment_timegan(splits_df, regime_key, ratio=1.0)

    model_tgan_r = PatchTST(
        h=TEST_MONTHS, input_size=min(12, REGIMES[regime_key]),
        patch_len=3, stride=1,
        n_heads=4, encoder_layers=3, hidden_size=128,
        linear_hidden_size=256, dropout=0.1, max_steps=500,
        batch_size=32, scaler_type="standard",
        random_seed=SEED+1, early_stop_patience_steps=-1,
        alias=f"PatchTST_TGAN_{regime_key}",
    )
    nf_tgan_r = NeuralForecast(models=[model_tgan_r], freq="MS")
    nf_tgan_r.fit(nf_train_tgan_r, val_size=min(6, REGIMES[regime_key]//2))

    preds_tgan_r = nf_tgan_r.predict(nf_train_tgan_r).reset_index()
    preds_tgan_r = preds_tgan_r[preds_tgan_r["unique_id"].isin(real_ids)]
    pred_col_t   = [c for c in preds_tgan_r.columns
                    if c not in ("unique_id","ds")][0]

    # ── Evaluate both ──
    real_mase, tgan_mase = [], []
    real_decomp, tgan_decomp = [], []

    for _, row in splits_df.iterrows():
        uid        = str(row["series_id"])
        y_true_sc  = row["test_sc"]
        y_true_raw = row["test_raw"]
        y_train_sc = get_regime_train(row["train_sc"], REGIMES[regime_key])

        # Real-Only
        fc_r = preds_real[preds_real["unique_id"] == uid]
        if not fc_r.empty:
            yp = fc_r[pred_col_r].values[:TEST_MONTHS]
            if len(yp) == TEST_MONTHS:
                m = mase(y_true_sc, yp, y_train_sc, seasonality=1)
                if not np.isnan(m):
                    real_mase.append(m)
                real_decomp.append(
                    mase_decomposed_raw(y_true_sc, yp, y_true_raw,
                                        y_train_sc, HIGH_THRESH_RAW))

        # TimeGAN
        fc_t = preds_tgan_r[preds_tgan_r["unique_id"] == uid]
        if not fc_t.empty:
            yp = fc_t[pred_col_t].values[:TEST_MONTHS]
            if len(yp) == TEST_MONTHS:
                m = mase(y_true_sc, yp, y_train_sc, seasonality=1)
                if not np.isnan(m):
                    tgan_mase.append(m)
                tgan_decomp.append(
                    mase_decomposed_raw(y_true_sc, yp, y_true_raw,
                                        y_train_sc, HIGH_THRESH_RAW))

    rd = pd.DataFrame(real_decomp)
    td = pd.DataFrame(tgan_decomp)

    regime_results[regime_key] = {
        "Real_Global" : float(np.median(real_mase)),
        "Real_Zero"   : float(rd["zero"].median()),
        "Real_High"   : float(rd["high"].median()),
        "TGAN_Global" : float(np.median(tgan_mase)),
        "TGAN_Zero"   : float(td["zero"].median()),
        "TGAN_High"   : float(td["high"].median()),
    }
    print(f"  Real  → Global:{np.median(real_mase):.4f}  "
          f"Zero:{rd['zero'].median():.4f}  "
          f"High:{rd['high'].median():.4f}")
    print(f"  TimeGAN → Global:{np.median(tgan_mase):.4f}  "
          f"Zero:{td['zero'].median():.4f}  "
          f"High:{td['high'].median():.4f}")

print("\n✅ Regime ablation complete.")

In [ ]:
# ============================================================
# CELL 45: Regime Ablation Summary
# ============================================================
regime_df = pd.DataFrame(regime_results).T
regime_df.index.name = "Regime"

print("=" * 65)
print("📊 CHECKPOINT 6 — Scarcity Regime Ablation (Real vs TimeGAN)")
print("=" * 65)
print(regime_df.round(4).to_string())

print("\n📌 TimeGAN Gain over Real-Only per regime:")
for regime in ["extreme", "moderate", "full"]:
    r = regime_df.loc[regime]
    print(f"\n  {regime} ({REGIMES[regime]} steps):")
    print(f"    Global: {r['Real_Global']:.4f} → {r['TGAN_Global']:.4f}  "
          f"Δ={r['TGAN_Global']-r['Real_Global']:+.4f} "
          f"{'✅' if r['TGAN_Global'] < r['Real_Global'] else '❌'}")
    print(f"    Zero  : {r['Real_Zero']:.4f} → {r['TGAN_Zero']:.4f}  "
          f"Δ={r['TGAN_Zero']-r['Real_Zero']:+.4f} "
          f"{'✅' if r['TGAN_Zero'] < r['Real_Zero'] else '❌'}")
    print(f"    High  : {r['Real_High']:.4f} → {r['TGAN_High']:.4f}  "
          f"Δ={r['TGAN_High']-r['Real_High']:+.4f} "
          f"{'✅' if r['TGAN_High'] < r['Real_High'] else '❌'}")

regime_df.to_csv("/content/results/checkpoint6_regime_ablation.csv")
print("\n✅ Checkpoint 6 saved.")